In [55]:
import requests
import pandas as pd
from io import StringIO

# TAHAP 1 PERSIAPAN KONFIGURASI API E-STAT

APP_ID = "69fbbb74fa35bfa86664ac12004c7b8b7296ea15"
STATS_DATA_ID = "0004007161"

url = f"""
http://api.e-stat.go.jp/rest/3.0/app/getSimpleStatsData
?appId={APP_ID}
&lang=J
&statsDataId={STATS_DATA_ID}
&metaGetFlg=Y
&cntGetFlg=N
&explanationGetFlg=Y
&annotationGetFlg=Y
&sectionHeaderFlg=1
&replaceSpChars=0
"""

# Membersihkan format URL
url = url.replace("\n", "").replace(" ", "")

print("Konfigurasi API berhasil disiapkan.")
print(f"ID Dataset e-Stat: {STATS_DATA_ID}")

Konfigurasi API berhasil disiapkan.
ID Dataset e-Stat: 0004007161


In [56]:
# TAHAP 2 PENGAMBILAN DATA DARI API E-STAT

print("Mulai menghubungkan sistem ke API e-Stat...")
response = requests.get(url)

print(f"Status koneksi API: {response.status_code}")

if response.status_code == 200:
    print("Koneksi API berhasil.")
else:
    print("Koneksi API gagal.")

Mulai menghubungkan sistem ke API e-Stat...
Status koneksi API: 200
Koneksi API berhasil.


In [57]:
# TAHAP 3 PEMBACAAN DATA MENTAH KE DATAFRAME

if response.status_code == 200:

    # Membaca respons teks dari API
    csv_data = StringIO(response.text)

    # Melewati metadata awal agar baris header dapat terbaca otomatis
    df_mentah = pd.read_csv(csv_data, skiprows=27)

    # --- TAMBAHAN PATH TARGET UNTUK DATA MENTAH ---
    path_data_mentah = "../data/raw/raw_gaji_prefecture.csv"
    
    # Mengekspor data mentah asli dari API ke folder raw
    df_mentah.to_csv(path_data_mentah, index=False, encoding='utf-8')
    # ----------------------------------------------

    print("\n=============================================")
    print("              TAHAP 3 SELESAI                ")
    print("=============================================")

    print("Data mentah berhasil dimuat ke DataFrame dan disimpan fisik.")
    print(f"Lokasi penyimpanan mentah: {path_data_mentah}")
    print(f"Jumlah data awal: {df_mentah.shape[0]} baris")
    print(f"Jumlah kolom awal: {df_mentah.shape[1]} kolom")
else:
    print("Gagal memuat data karena koneksi API bermasalah (Status bukan 200).")


              TAHAP 3 SELESAI                
Data mentah berhasil dimuat ke DataFrame dan disimpan fisik.
Lokasi penyimpanan mentah: ../data/raw/raw_gaji_prefecture.csv
Jumlah data awal: 18432 baris
Jumlah kolom awal: 13 kolom


In [58]:
# TAHAP 4 VERIFIKASI STRUKTUR DATA MENTAH

print("Menampilkan struktur kolom dataset gaji...\n")

print(df_mentah.columns.tolist())

Menampilkan struktur kolom dataset gaji...

['tab_code', '表章項目', 'cat01_code', '産業分類', 'cat02_code', '性別_基本', 'area_code', '地域', 'time_code', '時間軸（2020～2023）', 'unit', 'value', 'annotation']


In [59]:
# TAHAP 5 PENYARINGAN DATA PREFEKTUR

print("Mulai melakukan penyaringan data prefektur...")

# Menghapus data nasional dan menyisakan data tingkat prefektur
df_filter_prefektur = df_mentah[df_mentah['地域'] != '全国'].copy()

print("\nJumlah data setelah filter prefektur:")
print(f"{df_filter_prefektur.shape[0]} baris")

Mulai melakukan penyaringan data prefektur...

Jumlah data setelah filter prefektur:
18048 baris


In [60]:
# TAHAP 6 PENYARINGAN PERIODE TAHUN PENELITIAN

print("Mulai melakukan penyaringan periode tahun penelitian...")

# Menentukan rentang tahun penelitian
tahun_target = ['2020年', '2021年', '2022年', '2023年']

# Mengambil data sesuai tahun target
df_filter_tahun = df_filter_prefektur[
    df_filter_prefektur['時間軸（2020～2023）'].isin(tahun_target)
].copy()

print("\nTahun yang digunakan dalam penelitian:")
print(tahun_target)

print("\nJumlah data setelah filter tahun:")
print(f"{df_filter_tahun.shape[0]} baris")

Mulai melakukan penyaringan periode tahun penelitian...

Tahun yang digunakan dalam penelitian:
['2020年', '2021年', '2022年', '2023年']

Jumlah data setelah filter tahun:
18048 baris


In [61]:
# TAHAP 7 PENYARINGAN DATA BERDASARKAN JENIS KELAMIN

print("Mulai melakukan penyaringan kategori jenis kelamin...")

# Mengambil data gabungan laki-laki dan perempuan
df_filter_gender = df_filter_tahun[
    df_filter_tahun['性別_基本'] == '男女計'
].copy()

print("\nKategori gender yang digunakan:")
print("男女計 (Gabungan laki-laki dan perempuan)")

print("\nJumlah data setelah filter gender:")
print(f"{df_filter_gender.shape[0]} baris")

Mulai melakukan penyaringan kategori jenis kelamin...

Kategori gender yang digunakan:
男女計 (Gabungan laki-laki dan perempuan)

Jumlah data setelah filter gender:
6016 baris


In [62]:
# TAHAP 8 VERIFIKASI HASIL PENYARINGAN DATA

print("\n=============================================")
print("             TAHAP 8 SELESAI                 ")
print("=============================================")

print("Proses penyaringan data berhasil dilakukan.")
print(f"Total data hasil filter: {df_filter_gender.shape[0]} baris")

jumlah_terpangkas = df_mentah.shape[0] - df_filter_gender.shape[0]

print(f"Jumlah data yang dihapus: {jumlah_terpangkas} baris")

print("\n--- Sampel Data Setelah Penyaringan ---")

kolom_tampil = ['地域', '時間軸（2020～2023）', '性別_基本', 'value']

print(df_filter_gender[kolom_tampil].head())


             TAHAP 8 SELESAI                 
Proses penyaringan data berhasil dilakukan.
Total data hasil filter: 6016 baris
Jumlah data yang dihapus: 12416 baris

--- Sampel Data Setelah Penyaringan ---
    地域 時間軸（2020～2023） 性別_基本  value
4  北海道          2023年   男女計  288.8
5  北海道          2022年   男女計  339.6
6  北海道          2021年   男女計  284.0
7  北海道          2020年   男女計  308.4
8  青森県          2023年   男女計  295.3


In [63]:
import numpy as np

# TAHAP 9 PERSIAPAN DATA UNTUK PEMBERSIHAN

print("Mulai menyiapkan data untuk proses pembersihan...")

# Membuat salinan data hasil filtering
df_clean = df_filter_gender.copy()

print("Data berhasil disalin ke variabel baru.")
print(f"Jumlah data saat ini: {df_clean.shape[0]} baris")

Mulai menyiapkan data untuk proses pembersihan...
Data berhasil disalin ke variabel baru.
Jumlah data saat ini: 6016 baris


In [64]:
# TAHAP 10 PEMBERSIHAN NILAI KOSONG

print("Mulai membersihkan nilai kosong pada kolom value...")

# Menghapus spasi kosong pada data
df_clean['value'] = df_clean['value'].astype(str).str.strip()

# Mengganti simbol kosong menjadi NaN
df_clean['value'] = df_clean['value'].replace('-', np.nan)

# Menghapus baris yang memiliki nilai kosong
df_clean = df_clean.dropna(subset=['value'])

print("Pembersihan nilai kosong berhasil dilakukan.")
print(f"Jumlah data setelah pembersihan: {df_clean.shape[0]} baris")

Mulai membersihkan nilai kosong pada kolom value...
Pembersihan nilai kosong berhasil dilakukan.
Jumlah data setelah pembersihan: 6002 baris


In [65]:
# TAHAP 11 KONVERSI TIPE DATA NUMERIK

print("Mulai melakukan konversi tipe data numerik...")

# Mengubah tipe data kolom value menjadi float
df_clean['value'] = df_clean['value'].astype(float)

print("Konversi tipe data berhasil dilakukan.")

Mulai melakukan konversi tipe data numerik...
Konversi tipe data berhasil dilakukan.


In [66]:
# TAHAP 12 VERIFIKASI HASIL PEMBERSIHAN DATA

print("\n=============================================")
print("            TAHAP 12 SELESAI                 ")
print("=============================================")

print("Pembersihan dan konversi data berhasil dilakukan.")

print("\n--- Verifikasi Struktur Data ---")

print(df_clean[['value']].info())


            TAHAP 12 SELESAI                 
Pembersihan dan konversi data berhasil dilakukan.

--- Verifikasi Struktur Data ---
<class 'pandas.core.frame.DataFrame'>
Index: 6002 entries, 4 to 18047
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   value   6002 non-null   float64
dtypes: float64(1)
memory usage: 93.8 KB
None


In [67]:
# TAHAP 13 PERSIAPAN DATA KONVERSI NOMINAL GAJI

print("Mulai menyiapkan data untuk proses konversi nominal gaji...")

# Membuat salinan data hasil pembersihan
df_nominal = df_clean.copy()

print("Data berhasil disalin ke variabel baru.")
print(f"Jumlah data saat ini: {df_nominal.shape[0]} baris")

Mulai menyiapkan data untuk proses konversi nominal gaji...
Data berhasil disalin ke variabel baru.
Jumlah data saat ini: 6002 baris


In [68]:
# TAHAP 14 KONVERSI NILAI GAJI KE NOMINAL YEN

print("Mulai melakukan konversi nilai gaji ke nominal Yen asli...")

# Mengonversi nilai ribuan Yen menjadi nominal Yen penuh
df_nominal['gaji_nominal_yen'] = (
    df_nominal['value'] * 1000
).astype('int64')

print("Konversi nominal Yen berhasil dilakukan.")

Mulai melakukan konversi nilai gaji ke nominal Yen asli...
Konversi nominal Yen berhasil dilakukan.


In [69]:
# TAHAP 15 VERIFIKASI HASIL KONVERSI NOMINAL

print("\n=============================================")
print("            TAHAP 15 SELESAI                 ")
print("=============================================")

print("Kolom nominal gaji berhasil ditambahkan.")
print(f"Total data saat ini: {df_nominal.shape[0]} baris")

print("\n--- Sampel Data Hasil Konversi Nominal ---")

kolom_tampil = [
    '地域',
    '時間軸（2020～2023）',
    'value',
    'gaji_nominal_yen'
]

print(df_nominal[kolom_tampil].head())


            TAHAP 15 SELESAI                 
Kolom nominal gaji berhasil ditambahkan.
Total data saat ini: 6002 baris

--- Sampel Data Hasil Konversi Nominal ---
    地域 時間軸（2020～2023）  value  gaji_nominal_yen
4  北海道          2023年  288.8            288800
5  北海道          2022年  339.6            339600
6  北海道          2021年  284.0            284000
7  北海道          2020年  308.4            308400
8  青森県          2023年  295.3            295300


In [70]:
# TAHAP 16 PERSIAPAN DATA TRANSLASI

print("Mulai menyiapkan data untuk proses translasi...")

# Membuat salinan data hasil konversi nominal
df_translate = df_nominal.copy()

print("Data berhasil disalin ke variabel baru.")
print(f"Jumlah data saat ini: {df_translate.shape[0]} baris")

Mulai menyiapkan data untuk proses translasi...
Data berhasil disalin ke variabel baru.
Jumlah data saat ini: 6002 baris


In [71]:
# TAHAP 17 TRANSLASI NAMA PREFEKTUR

print("Mulai melakukan translasi nama prefektur...")

# Kamus translasi prefektur Jepang
dict_prefektur = {
    '北海道': 'Hokkaido', '青森県': 'Aomori', '岩手県': 'Iwate', '宮城県': 'Miyagi',
    '秋田県': 'Akita', '山形県': 'Yamagata', '福島県': 'Fukushima', '茨城県': 'Ibaraki',
    '栃木県': 'Tochigi', '群馬県': 'Gunma', '埼玉県': 'Saitama', '千葉県': 'Chiba',
    '東京都': 'Tokyo', '神奈川県': 'Kanagawa', '新潟県': 'Niigata', '富山県': 'Toyama',
    '石川県': 'Ishikawa', '福井県': 'Fukui', '山梨県': 'Yamanashi', '長野県': 'Nagano',
    '岐阜県': 'Gifu', '静岡県': 'Shizuoka', '愛知県': 'Aichi', '三重県': 'Mie',
    '滋賀県': 'Shiga', '京都府': 'Kyoto', '大阪府': 'Osaka', '兵庫県': 'Hyogo',
    '奈良県': 'Nara', '和歌山県': 'Wakayama', '鳥取県': 'Tottori', '島根県': 'Shimane',
    '岡山県': 'Okayama', '広島県': 'Hiroshima', '山口県': 'Yamaguchi', '徳島県': 'Tokushima',
    '香川県': 'Kagawa', '愛媛県': 'Ehime', '高知県': 'Kochi', '福岡県': 'Fukuoka',
    '佐賀県': 'Saga', '長崎県': 'Nagasaki', '熊本県': 'Kumamoto', '大分県': 'Oita',
    '宮崎県': 'Miyazaki', '鹿児島県': 'Kagoshima', '沖縄県': 'Okinawa'
}

# Melakukan translasi nama prefektur
df_translate['prefektur_en'] = (
    df_translate['地域']
    .map(dict_prefektur)
    .fillna(df_translate['地域'])
)

print("Translasi nama prefektur berhasil dilakukan.")

Mulai melakukan translasi nama prefektur...
Translasi nama prefektur berhasil dilakukan.


In [72]:
# TAHAP 18 TRANSLASI KATEGORI INDUSTRI

print("Mulai melakukan translasi kategori industri...")

# Fungsi translasi industri berdasarkan kode klasifikasi e-Stat
def translate_industri(text):

    text = str(text).strip()

    mapping = {
        'Ｃ': {
            'id': 'Pertambangan dan Penggalian',
            'en': 'Mining and Quarrying'
        },
        'Ｄ': {
            'id': 'Konstruksi',
            'en': 'Construction'
        },
        'Ｅ': {
            'id': 'Manufaktur',
            'en': 'Manufacturing'
        },
        'Ｆ': {
            'id': 'Pemasokan Listrik, Gas, dan Air',
            'en': 'Electricity, Gas, and Water Supply'
        },
        'Ｇ': {
            'id': 'Informasi dan Komunikasi',
            'en': 'Information and Communications'
        },
        'Ｈ': {
            'id': 'Transportasi dan Pos',
            'en': 'Transport and Postal Activities'
        },
        'Ｉ': {
            'id': 'Perdagangan Besar dan Eceran',
            'en': 'Wholesale and Retail Trade'
        },
        'Ｊ': {
            'id': 'Keuangan dan Asuransi',
            'en': 'Finance and Insurance'
        },
        'Ｋ': {
            'id': 'Real Estat dan Penyewaan Barang',
            'en': 'Real Estate and Goods Rental'
        },
        'Ｌ': {
            'id': 'Layanan Ilmiah/Akademik dan Profesional',
            'en': 'Scientific, Academic and Professional Services'
        },
        'Ｍ': {
            'id': 'Perhotelan dan Restoran (Hospitality)',
            'en': 'Accommodation and Food Services'
        },
        'Ｎ': {
            'id': 'Layanan Hidup, Hiburan, dan Rekreasi',
            'en': 'Living-related, Amusement and Recreation Services'
        },
        'Ｏ': {
            'id': 'Layanan Pendidikan dan Pembelajaran',
            'en': 'Education and Learning Support Services'
        },
        'Ｐ': {
            'id': 'Kesehatan dan Kesejahteraan Sosial',
            'en': 'Medical, Health Care and Welfare'
        },
        'Ｑ': {
            'id': 'Layanan Gabungan',
            'en': 'Compound Services'
        },
        'Ｒ': {
            'id': 'Layanan Lainnya yang Tidak Diklasifikasikan',
            'en': 'Services, N.E.C.'
        }
    }

    huruf_depan = text[0] if len(text) > 0 else ''

    if huruf_depan in mapping:
        return mapping[huruf_depan]['id'], mapping[huruf_depan]['en']

    return text, text

# Menjalankan proses translasi industri
hasil_translasi = df_translate['産業分類'].apply(translate_industri)

# Menambahkan hasil translasi ke dataframe
df_translate['industri_id'] = [item[0] for item in hasil_translasi]
df_translate['industri_en'] = [item[1] for item in hasil_translasi]

print("Translasi kategori industri berhasil dilakukan.")

Mulai melakukan translasi kategori industri...
Translasi kategori industri berhasil dilakukan.


In [73]:
# TAHAP 19 TRANSLASI KATEGORI GAJI

print("Mulai melakukan translasi kategori gaji...")

# Translasi kategori gaji ke Bahasa Indonesia
df_translate['kategori_gaji_id'] = (
    df_translate['表章項目']
    .replace('所定内給与額', 'Gaji Pokok Bulanan')
)

print("Translasi kategori gaji berhasil dilakukan.")

Mulai melakukan translasi kategori gaji...
Translasi kategori gaji berhasil dilakukan.


In [74]:
# TAHAP 20 VERIFIKASI HASIL TRANSLASI

print("\n=============================================")
print("            TAHAP 20 SELESAI                 ")
print("=============================================")

print("Proses translasi data berhasil dilakukan.")
print(f"Total data saat ini: {df_translate.shape[0]} baris")

print("\n--- Sampel Data Hasil Translasi ---")

kolom_tampil = [
    '地域',
    'prefektur_en',
    '産業分類',
    'industri_id',
    'kategori_gaji_id',
    'gaji_nominal_yen'
]

print(df_translate[kolom_tampil].head(3).to_string())


            TAHAP 20 SELESAI                 
Proses translasi data berhasil dilakukan.
Total data saat ini: 6002 baris

--- Sampel Data Hasil Translasi ---
    地域 prefektur_en            産業分類                  industri_id    kategori_gaji_id  gaji_nominal_yen
4  北海道     Hokkaido  Ｃ 鉱業，採石業，砂利採取業  Pertambangan dan Penggalian  Gaji Pokok Bulanan            288800
5  北海道     Hokkaido  Ｃ 鉱業，採石業，砂利採取業  Pertambangan dan Penggalian  Gaji Pokok Bulanan            339600
6  北海道     Hokkaido  Ｃ 鉱業，採石業，砂利採取業  Pertambangan dan Penggalian  Gaji Pokok Bulanan            284000


In [75]:
# TAHAP 21 PERSIAPAN DATA ATRIBUT SAW

print("Mulai menyiapkan data untuk penambahan atribut SAW...")

# Membuat salinan data hasil translasi
df_saw = df_translate.copy()

print("Data berhasil disalin ke variabel baru.")
print(f"Jumlah data saat ini: {df_saw.shape[0]} baris")

Mulai menyiapkan data untuk penambahan atribut SAW...
Data berhasil disalin ke variabel baru.
Jumlah data saat ini: 6002 baris


In [76]:
# TAHAP 22 PENAMBAHAN ATRIBUT KRITERIA SAW

print("Mulai menambahkan atribut kriteria SAW...")

# Menambahkan atribut tipe kriteria
df_saw['tipe_kriteria'] = 'Benefit'

print("Atribut tipe kriteria berhasil ditambahkan.")

Mulai menambahkan atribut kriteria SAW...
Atribut tipe kriteria berhasil ditambahkan.


In [77]:
# TAHAP 23 VERIFIKASI HASIL ATRIBUT SAW

print("\n=============================================")
print("            TAHAP 23 SELESAI                 ")
print("=============================================")

print("Penambahan atribut SAW berhasil dilakukan.")
print(f"Total data saat ini: {df_saw.shape[0]} baris")

print("\n--- Sampel Data Hasil Atribut SAW ---")

kolom_tampil = [
    'prefektur_en',
    'industri_id',
    'gaji_nominal_yen',
    'tipe_kriteria'
]

print(df_saw[kolom_tampil].head(3))


            TAHAP 23 SELESAI                 
Penambahan atribut SAW berhasil dilakukan.
Total data saat ini: 6002 baris

--- Sampel Data Hasil Atribut SAW ---
  prefektur_en                  industri_id  gaji_nominal_yen tipe_kriteria
4     Hokkaido  Pertambangan dan Penggalian            288800       Benefit
5     Hokkaido  Pertambangan dan Penggalian            339600       Benefit
6     Hokkaido  Pertambangan dan Penggalian            284000       Benefit


In [78]:
# TAHAP 24 PERSIAPAN DATA FINAL UNTUK AGREGASI

print("Mulai menyiapkan data final untuk proses agregasi...")

# Mengambil salinan data hasil atribut SAW
df_final_process = df_saw.copy()

print("Data berhasil disiapkan.")
print(f"Jumlah data saat ini: {df_final_process.shape[0]} baris")

Mulai menyiapkan data final untuk proses agregasi...
Data berhasil disiapkan.
Jumlah data saat ini: 6002 baris


In [79]:
# TAHAP 25 STANDARISASI KATEGORI GAJI UNTUK PIVOTING

print("Mulai melakukan standarisasi kategori gaji...")

# Mengubah nama kategori menjadi format teknis yang lebih rapi
df_final_process['kategori_gaji_id'] = df_final_process['kategori_gaji_id'].replace({
    'Gaji Pokok Bulanan': 'gaji_pokok_bulanan',
    '年間賞与その他特別給与額': 'bonus_tahunan_rata_rata'
})

print("Standarisasi kategori gaji berhasil dilakukan.")

print("\nKategori yang tersedia:")
print(df_final_process['kategori_gaji_id'].unique())

Mulai melakukan standarisasi kategori gaji...
Standarisasi kategori gaji berhasil dilakukan.

Kategori yang tersedia:
['gaji_pokok_bulanan' 'bonus_tahunan_rata_rata']


In [80]:
# TAHAP 26 AGREGASI RATA-RATA GAJI PER PREFEKTUR DAN INDUSTRI

print("Mulai menghitung rata-rata gaji berdasarkan prefektur dan industri...")

# Menghitung rata-rata nominal gaji
df_grouped = df_final_process.groupby(
    [
        'prefektur_en',
        'industri_id',
        'kategori_gaji_id',
        'tipe_kriteria'
    ],
    as_index=False
).agg({
    'gaji_nominal_yen': 'mean'
})

print("Proses agregasi berhasil dilakukan.")
print(f"Total data hasil agregasi: {df_grouped.shape[0]} baris")

print("\n--- Sampel Hasil Agregasi ---")
print(df_grouped.head(5))

Mulai menghitung rata-rata gaji berdasarkan prefektur dan industri...
Proses agregasi berhasil dilakukan.
Total data hasil agregasi: 1504 baris

--- Sampel Hasil Agregasi ---
  prefektur_en                         industri_id         kategori_gaji_id  \
0        Aichi            Informasi dan Komunikasi  bonus_tahunan_rata_rata   
1        Aichi            Informasi dan Komunikasi       gaji_pokok_bulanan   
2        Aichi  Kesehatan dan Kesejahteraan Sosial  bonus_tahunan_rata_rata   
3        Aichi  Kesehatan dan Kesejahteraan Sosial       gaji_pokok_bulanan   
4        Aichi               Keuangan dan Asuransi  bonus_tahunan_rata_rata   

  tipe_kriteria  gaji_nominal_yen  
0       Benefit         1325875.0  
1       Benefit          365800.0  
2       Benefit          827925.0  
3       Benefit          308150.0  
4       Benefit         1283725.0  


In [81]:
# TAHAP 27 TRANSFORMASI STRUKTUR DATA DENGAN PIVOTING

print("Mulai melakukan transformasi struktur data horizontal...")

# Mengubah kategori gaji dari baris menjadi kolom
df_pivot = df_grouped.pivot(
    index=[
        'prefektur_en',
        'industri_id',
        'tipe_kriteria'
    ],
    columns='kategori_gaji_id',
    values='gaji_nominal_yen'
).reset_index()

print("Transformasi pivot berhasil dilakukan.")
print(f"Total data hasil pivot: {df_pivot.shape[0]} baris")

print("\n--- Sampel Data Hasil Pivot ---")
print(df_pivot.head(5))

Mulai melakukan transformasi struktur data horizontal...
Transformasi pivot berhasil dilakukan.
Total data hasil pivot: 752 baris

--- Sampel Data Hasil Pivot ---
kategori_gaji_id prefektur_en                         industri_id  \
0                       Aichi            Informasi dan Komunikasi   
1                       Aichi  Kesehatan dan Kesejahteraan Sosial   
2                       Aichi               Keuangan dan Asuransi   
3                       Aichi                          Konstruksi   
4                       Aichi                    Layanan Gabungan   

kategori_gaji_id tipe_kriteria  bonus_tahunan_rata_rata  gaji_pokok_bulanan  
0                      Benefit                1325875.0            365800.0  
1                      Benefit                 827925.0            308150.0  
2                      Benefit                1283725.0            369300.0  
3                      Benefit                1207100.0            347750.0  
4                      Benefit  

In [82]:
# TAHAP 28 PEMBERSIHAN DAN PENYESUAIAN NILAI AKHIR

print("Mulai melakukan pembersihan data akhir...")

# Mengisi nilai kosong dengan 0
df_pivot = df_pivot.fillna(0)

# Membulatkan nilai numerik menjadi integer
df_pivot['gaji_pokok_bulanan'] = (
    df_pivot['gaji_pokok_bulanan']
    .round()
    .astype('int64')
)

df_pivot['bonus_tahunan_rata_rata'] = (
    df_pivot['bonus_tahunan_rata_rata']
    .round()
    .astype('int64')
)

print("Pembersihan data akhir berhasil dilakukan.")

print("\n--- Sampel Data Setelah Dibersihkan ---")
print(
    df_pivot[
        [
            'prefektur_en',
            'industri_id',
            'gaji_pokok_bulanan',
            'bonus_tahunan_rata_rata'
        ]
    ].head(5)
)

Mulai melakukan pembersihan data akhir...
Pembersihan data akhir berhasil dilakukan.

--- Sampel Data Setelah Dibersihkan ---
kategori_gaji_id prefektur_en                         industri_id  \
0                       Aichi            Informasi dan Komunikasi   
1                       Aichi  Kesehatan dan Kesejahteraan Sosial   
2                       Aichi               Keuangan dan Asuransi   
3                       Aichi                          Konstruksi   
4                       Aichi                    Layanan Gabungan   

kategori_gaji_id  gaji_pokok_bulanan  bonus_tahunan_rata_rata  
0                             365800                  1325875  
1                             308150                   827925  
2                             369300                  1283725  
3                             347750                  1207100  
4                             309375                  1165425  


In [83]:
# TAHAP 29 PENGURUTAN DATA FINAL

print("Mulai mengurutkan data final...")

# Mengurutkan data berdasarkan prefektur dan industri
df_final = df_pivot.sort_values(
    by=[
        'prefektur_en',
        'industri_id'
    ]
).reset_index(drop=True)

print("Pengurutan data berhasil dilakukan.")
print(f"Total data final: {df_final.shape[0]} baris")

Mulai mengurutkan data final...
Pengurutan data berhasil dilakukan.
Total data final: 752 baris


In [84]:
# TAHAP 30 EKSPOR DATASET FINAL KE FILE CSV

print("Mulai mengekspor dataset final ke file CSV...")


# --- PERUBAHAN PATH TARGET UNTUK DATA FINAL ---
nama_file_csv = "../data/processed/dataset_gaji_final.csv"


# Ekspor ke CSV
df_final.to_csv(
    nama_file_csv,
    index=False,
    encoding='utf-8'
)

print("\n=============================================")
print("              TAHAP 30 SELESAI               ")
print("=============================================")

print("Dataset final berhasil diekspor.")
print(f"Nama file / Jalur: {nama_file_csv}")
print(f"Total data final: {df_final.shape[0]} baris")

print("\n--- Sampel Dataset Final ---")

kolom_final = [
    'prefektur_en',
    'industri_id',
    'gaji_pokok_bulanan',
    'bonus_tahunan_rata_rata',
    'tipe_kriteria'
]

print(df_final[kolom_final].head(5))

Mulai mengekspor dataset final ke file CSV...

              TAHAP 30 SELESAI               
Dataset final berhasil diekspor.
Nama file / Jalur: ../data/processed/dataset_gaji_final.csv
Total data final: 752 baris

--- Sampel Dataset Final ---
kategori_gaji_id prefektur_en                         industri_id  \
0                       Aichi            Informasi dan Komunikasi   
1                       Aichi  Kesehatan dan Kesejahteraan Sosial   
2                       Aichi               Keuangan dan Asuransi   
3                       Aichi                          Konstruksi   
4                       Aichi                    Layanan Gabungan   

kategori_gaji_id  gaji_pokok_bulanan  bonus_tahunan_rata_rata tipe_kriteria  
0                             365800                  1325875       Benefit  
1                             308150                   827925       Benefit  
2                             369300                  1283725       Benefit  
3                            

In [85]:
print(df_final.columns.tolist())

['prefektur_en', 'industri_id', 'tipe_kriteria', 'bonus_tahunan_rata_rata', 'gaji_pokok_bulanan']
